<a href="https://colab.research.google.com/github/acamogh/Gradio_Quiz_App/blob/main/Google_Cloud_Generative_AI_Leader_Practice_Exam_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
# =====================================================================
# CELL 1: ENVIRONMENT CONFIGURATION, MOUNT DRIVE & PARSING (IMPROVED)
# =====================================================================
print("📦 Installing packages...")
import sys
import subprocess
import os
import glob
import json
import random

subprocess.run([sys.executable, "-m", "pip", "install", "--ignore-installed", "-q", "-U", "groq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "langchain-community", "pypdf", "tiktoken", "gradio",
                "langchain-text-splitters"], check=True)

print("✅ Libraries installed successfully.")

import gradio as gr
from groq import Groq
from google.colab import userdata, drive
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Mount Drive
print("\n🗄️ Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

DRIVE_APP_DIR = "/content/drive/MyDrive/GenAI_Study_App"
os.makedirs(DRIVE_APP_DIR, exist_ok=True)
LOG_FILE_PATH = os.path.join(DRIVE_APP_DIR, "quiz_mistakes_log.json")
print(f"📁 Log Path: {LOG_FILE_PATH}")

print("\n📂 Scanning for PDF files...")
pdf_files = glob.glob("*.pdf")

if not pdf_files:
    raise FileNotFoundError("⚠️ No PDF files found! Upload your 8 PDFs and rerun this cell.")

print(f"📄 Found {len(pdf_files)} PDFs. Processing with smart chunking...")

# ====================== SMART CHUNKING ======================
KNOWLEDGE_CHUNKS = []

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len
)

for pdf in pdf_files:
    try:
        loader = PyPDFLoader(pdf)
        pages = loader.load()
        split_docs = text_splitter.split_documents(pages)

        for doc in split_docs:
            text = doc.page_content.strip()
            if text and len(text) > 100:
                page_num = doc.metadata.get("page", 0) + 1
                KNOWLEDGE_CHUNKS.append({
                    "source": pdf,
                    "page": page_num,
                    "content": text
                })
    except Exception as e:
        print(f"   Error reading {pdf}: {e}")

print(f"✅ Loaded {len(KNOWLEDGE_CHUNKS)} smart chunks from {len(pdf_files)} PDFs.")

📦 Installing packages...
✅ Libraries installed successfully.

🗄️ Connecting to Google Drive...
Mounted at /content/drive
📁 Log Path: /content/drive/MyDrive/GenAI_Study_App/quiz_mistakes_log.json

📂 Scanning for PDF files...
📄 Found 8 PDFs. Processing with smart chunking...
✅ Loaded 946 smart chunks from 8 PDFs.


In [36]:
# =====================================================================
# CELL 2: COMPLETE QUIZ ENGINE, DRIVER LOG MATRIX & GRADIO INTERFACE
# =====================================================================
import os
import json
import random
import gradio as gr
from groq import Groq

class MobileQuizEngine:
    def __init__(self, knowledge_chunks, log_path):
        self.chunks = knowledge_chunks
        self.log_path = log_path
        self.current_quiz = None
        self.quota_string = "🎯 RPD Left: `Pending...` | ⏳ Rolling TPM Window: `Pending...`"

        self.model_id = 'llama-3.3-70b-versatile'
        self.backup_model_id = 'llama-3.1-8b-instant'

        self.incorrect_log = self._load_history_from_drive()

        try:
            from google.colab import userdata
            api_key = userdata.get('GROQ_API_KEY')
            self.groq_client = Groq(api_key=api_key)
        except Exception:
            raise ValueError("❌ GROQ_API_KEY missing. Set it in Colab's Secrets menu.")

    def _load_history_from_drive(self):
        if os.path.exists(self.log_path):
            try:
                with open(self.log_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    print(f"💾 Restored {len(data)} past conceptual weak areas from Google Drive!")
                    return data
            except Exception as e:
                print(f"⚠️ Could not parse log file: {e}")
                return []
        print("📝 No existing log found. Starting fresh.")
        return []

    def _write_history_to_drive(self):
        try:
            with open(self.log_path, 'w', encoding='utf-8') as f:
                json.dump(self.incorrect_log, f, indent=4, ensure_ascii=False)
        except Exception as e:
            print(f"🚨 Failed to save log to Drive: {e}")

    def generate_question(self):
        review_context = ""
        if self.incorrect_log and len(self.incorrect_log) % 3 == 0:
            past_mistake = self.incorrect_log[-1]
            review_context = f"""
            CRITICAL REVIEW CONTEXT: The student previously failed a question on concept: '{past_mistake['concept']}'.
            Do NOT reuse the old question. Paraphrase heavily and test from a different angle.
            """

        # RICH CONTEXT: Pick 3 consecutive smart chunks
        if len(self.chunks) >= 3:
            start_idx = random.randint(0, len(self.chunks) - 3)
            selected_chunks = self.chunks[start_idx:start_idx + 3]
        else:
            selected_chunks = self.chunks

        dynamic_context = ""
        for idx, chunk in enumerate(selected_chunks):
            dynamic_context += f"\n--- Context Source {idx+1}: File '{chunk['source']}', Page ~{chunk['page']} ---\n"
            dynamic_context += chunk['content'] + "\n"

        system_instruction = f"""
        You are an expert exam simulator for the Google Cloud Generative AI Leader certification.

        Knowledge Base Material:
        {dynamic_context}

        Instructions:
        1. NEVER copy exact sentences from the source.
        2. Generate one highly challenging multiple-choice question.
        3. Provide exactly four options (A, B, C, D) with ONE correct answer.
        4. Identify the main source file and approximate page.

        {review_context}

        Return data STRICTLY as JSON:
        {{
            "question": "...",
            "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
            "correct_answer": "A/B/C/D",
            "concept": "Technical principle being tested",
            "explanation": "Detailed explanation...",
            "source_file": "filename.pdf",
            "source_page": "page number"
        }}
        """

        try:
            raw_response = self.groq_client.chat.completions.with_raw_response.create(
                model=self.model_id,
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": "Generate the next certification exam question."}
                ],
                response_format={"type": "json_object"},
                temperature=0.8
            )

            headers = raw_response.headers
            rem_req = headers.get("x-ratelimit-remaining-requests", "N/A")
            rem_tok = headers.get("x-ratelimit-remaining-tokens", "N/A")
            self.quota_string = f"🧠 **Active:** `{self.model_id}` | 🎯 **RPD:** `{rem_req}` | ⏳ **TPM:** `{rem_tok}`"

            completion = raw_response.parse()
            raw_content = completion.choices[0].message.content
            self.current_quiz = json.loads(raw_content)
            return self.current_quiz

        except Exception as e:
            print(f"⚡ Primary model issue: {e}. Using fallback...")
            # Fallback code remains the same...
            response = self.groq_client.chat.completions.create(
                model=self.backup_model_id,
                messages=[{"role": "system", "content": system_instruction},
                         {"role": "user", "content": "Generate the next certification exam question."}],
                response_format={"type": "json_object"},
                temperature=0.7
            )
            self.quota_string = f"⚠️ **Fallback:** `{self.backup_model_id}`"
            raw_content = response.choices[0].message.content
            self.current_quiz = json.loads(raw_content)
            return self.current_quiz

    def log_failure(self):
        if self.current_quiz:
            self.incorrect_log.append({
                "concept": self.current_quiz.get("concept", "General Concept"),
                "question_text": self.current_quiz.get("question", "")[:300],
                "source_file": self.current_quiz.get("source_file", "Unknown"),
                "source_page": self.current_quiz.get("source_page", "Unknown")
            })
            self._write_history_to_drive()

    def get_summary(self):
        if not self.incorrect_log:
            return "🎉 **Perfect record! No weak areas logged yet.**"

        summary = "### 📊 Conceptual Weak Areas Dashboard\n\n"
        for idx, item in enumerate(self.incorrect_log, 1):
            summary += f"{idx}. **{item['concept']}**\n"
            summary += f"   📄 `{item['source_file']}` | 📖 Page `{item['source_page']}`\n---\n"
        return summary

# Instantiate engine
quiz_engine = MobileQuizEngine(KNOWLEDGE_CHUNKS, LOG_FILE_PATH)

# ====================== GRADIO INTERFACE (unchanged) ======================
def load_new_question():
    data = quiz_engine.generate_question()
    if not data:
        return "❌ Error generating question. Try again.", [], "", "", quiz_engine.quota_string

    q_text = f"### Question:\n{data['question']}"
    meta_text = f"📖 **Source:** `{data.get('source_file', 'Unknown')}` (Page `{data.get('source_page', 'N/A')}`)"
    choices_list = [f"{k}: {v}" for k, v in data['options'].items()]

    return q_text, gr.Radio(choices=choices_list, value=None), "", meta_text, quiz_engine.quota_string

def evaluate_selection(choice):
    if not quiz_engine.current_quiz or not choice:
        return "⚠️ Please load a question and select an answer first."

    user_letter = choice.split(":")[0].strip().upper()
    correct_letter = quiz_engine.current_quiz['correct_answer'].strip().upper()

    if user_letter == correct_letter:
        score_text = "### ✅ CORRECT!\n\n"
    else:
        score_text = f"### ❌ INCORRECT. Correct answer: **({correct_letter})**\n\n"
        quiz_engine.log_failure()

    return f"{score_text}**Explanation:**\n{quiz_engine.current_quiz['explanation']}"

def view_summary_panel():
    return quiz_engine.get_summary()

# ... [Rest of your Gradio UI code remains exactly the same] ...

custom_font_css = """
@import url('https://fonts.googleapis.com/css2?family=Geist:wght@300;400;500;600;700&display=swap');
body, .gradio-container, input, button, select, textarea, span, p, h1, h2, h3, label {
    font-family: 'Google Sans', 'Product Sans', 'Geist', -apple-system, BlinkMacSystemFont, sans-serif !important;
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_font_css) as mobile_app:
    gr.Markdown("# 🧠 Google Cloud GenAI Leader App Hub")

    with gr.Group():
        question_display = gr.Markdown("### Tap '➡️ Fetch Next Question' to begin!")
        metadata_tracker = gr.Markdown("")

    choice_radio = gr.Radio(choices=[], label="👇 Select your answer option:")
    submit_btn = gr.Button("🔒 Submit Selection", variant="primary")
    feedback_display = gr.Markdown("")

    with gr.Row():
        next_btn = gr.Button("➡️ Fetch Next Question", variant="secondary")
        summary_btn = gr.Button("📊 Weak Summary Dashboard", variant="stop")

    gr.Markdown("---")
    summary_display = gr.Markdown("")

    gr.Markdown("---")
    quota_display = gr.Markdown("")

    next_btn.click(load_new_question, inputs=[], outputs=[question_display, choice_radio, feedback_display, metadata_tracker, quota_display])
    submit_btn.click(evaluate_selection, inputs=[choice_radio], outputs=[feedback_display])
    summary_btn.click(view_summary_panel, inputs=[], outputs=[summary_display])

mobile_app.launch(inline=False, share=True, debug=False)

📝 No existing log found. Starting fresh.


/tmp/ipykernel_4435/4166581619.py:194: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_font_css) as mobile_app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20a7e2afe6183da916.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
